# DREX-UNIFIED POC — Lightning AI

**Platform:** Lightning AI Studio (Jupyter-compatible, persistent workspace)

Run any one of the 5 POC sprints — Sprint 5 (scale run, 50k steps) is the primary
use-case for Lightning AI because sessions are not time-limited.

## Platform assignment matrix

| Platform | Notebook | Recommended sprints | Est. time (T4) | Cost |
|---|---|---|---|---|
| Google Colab Free (T4) | `colab_drex_poc.ipynb` | 1, 2, 3, 4 | ~25 min/sprint | Free |
| Kaggle Free (P100) | `kaggle_drex_poc.ipynb` | 1, 2, 3, 4 | ~30 min/sprint | Free |
| Lightning AI | `lightning_drex_poc.ipynb` | All, esp. 5 | ~90 min (S5) | ~$0.20–$0.50 |

> **Quick start:**
> 1. Open a Lightning AI Studio (or TeamSpace).
> 2. Open this notebook (`notebooks/platforms/lightning_drex_poc.ipynb`).
> 3. Set `SPRINT` below and click **Run All**.
> Results persist in `/teamspace/studios/this_studio/drex_poc/`.

## Sprint reference

| Sprint | Config | Seeds | Steps | Gate |
|---|---|---|---|---|
| 1 | Baseline transformer | 42, 43, 44 | 10k | val_ppl < 2.5 |
| 2 | + Mamba SSM backbone | 42, 43, 44 | 10k | ≤ Sprint 1 + 0.20 |
| 3 | + ESN episodic memory | 42, 43, 44 | 10k | < Sprint 2 |
| 4 | + HDC encoder | 42, 43, 44 | 10k | ≤ Sprint 3 |
| 5 | Scale (d=256, 8L, 512-seg) | 42 | 50k | val_ppl ≤ S1 + passkey ≥ 2× S1 |

In [ ]:
# ── Sprint selector ──────────────────────────────────────────────────────────
# Change SPRINT to 1, 2, 3, 4, or 5 before running.
SPRINT = 5        # Sprint 5 is the primary Lightning use-case (50k steps)
SEEDS = [42, 43, 44]   # Sprint 5 always uses [42] regardless of this setting
BATCH_SIZE = 32   # 32 is safe on an L4/T4 (16 GB VRAM) for d=128/256
SKIP_SETUP = False  # Set True to skip git clone + pip install (re-runs only)

print(f"Will run Sprint {SPRINT} with seeds {SEEDS}, batch_size={BATCH_SIZE}")
print(f"SKIP_SETUP={SKIP_SETUP}")

In [ ]:
# ── Environment setup ────────────────────────────────────────────────────────
import os
import sys
import subprocess

# Lightning AI: workspace persists across sessions
RESULTS_ROOT = "/teamspace/studios/this_studio/drex_poc"
os.makedirs(RESULTS_ROOT, exist_ok=True)

if not SKIP_SETUP:
    # Lightning AI images ship PyTorch; install only extra deps
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-U",
         "datasets", "safetensors", "tqdm"],
        check=True,
    )
    # causal-conv1d is required for Mamba (Sprints 2-5)
    try:
        import causal_conv1d  # noqa: F401
    except ImportError:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", "causal-conv1d>=1.4.0"],
            check=True,
        )
else:
    print("SKIP_SETUP=True — skipping pip install.")

print("Setup complete.")
print(f"RESULTS_ROOT: {RESULTS_ROOT}")

In [ ]:
# ── Clone / update the drex repo ─────────────────────────────────────────────
REPO_URL = "https://github.com/squishai/drex.git"
REPO_DIR = "/teamspace/studios/this_studio/drex"

if not SKIP_SETUP:
    if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
        subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
    else:
        subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
else:
    print("SKIP_SETUP=True — skipping git clone/pull.")

sys.path.insert(0, os.path.join(REPO_DIR, "python"))
print(f"Repo at: {REPO_DIR}")

In [ ]:
# ── GPU / device check ───────────────────────────────────────────────────────
import torch

if torch.cuda.is_available():
    device_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024 ** 3
    print(f"GPU: {device_name}  ({vram_gb:.1f} GB VRAM)")
    DEVICE = "cuda"
else:
    print("WARNING: No CUDA GPU found.")
    print("Add a GPU in your Lightning AI Studio settings and restart the kernel.")
    DEVICE = "cpu"

print(f"Device: {DEVICE}")

In [ ]:
# ── Run the sprint via run_poc_cloud.py ──────────────────────────────────────
# Sprint 5 always uses seed 42 only (50k-step scale run)
seeds_for_sprint = [42] if SPRINT == 5 else SEEDS

cmd = [
    sys.executable,
    os.path.join(REPO_DIR, "scripts", "poc", "run_poc_cloud.py"),
    "--sprint", str(SPRINT),
    "--seeds", *[str(s) for s in seeds_for_sprint],
    "--out-dir", RESULTS_ROOT,
    "--batch-size", str(BATCH_SIZE),
    "--device", DEVICE,
    "--repo-root", REPO_DIR,
]

print("Running:", " ".join(cmd))
print()

result = subprocess.run(cmd, cwd=REPO_DIR)

if result.returncode != 0:
    raise RuntimeError(f"run_poc_cloud.py exited with code {result.returncode}")
print("\nSprint complete.")

In [ ]:
# ── Results summary + sprint gate ─────────────────────────────────────────────
import json
from pathlib import Path

_SPRINT_NAMES = {1: "exp_poc_a", 2: "exp_poc_b", 3: "exp_poc_c", 4: "exp_poc_d", 5: "exp_poc_e"}
_SPRINT_GATES = {
    1: ("val_ppl < 2.5",                       lambda p, c: p < 2.5),
    2: ("median val_ppl <= Sprint-1 + 0.20",   lambda p, c: p <= c.get("s1_ppl", 9999) + 0.20),
    3: ("median val_ppl < Sprint-2 median",    lambda p, c: p < c.get("s2_ppl", 9999)),
    4: ("median val_ppl <= Sprint-3 median",   lambda p, c: p <= c.get("s3_ppl", 9999)),
    5: ("val_ppl <= Sprint-1 median",          lambda p, c: p <= c.get("s1_ppl", 9999)),
}

sprint_name = _SPRINT_NAMES[SPRINT]
summary_file = Path(RESULTS_ROOT) / f"{sprint_name}_summary.json"

# Load prior sprint medians for comparative gate checks
_sprint_medians: dict = {}
for _s, _n in _SPRINT_NAMES.items():
    _f = Path(RESULTS_ROOT) / f"{_n}_summary.json"
    if _f.exists():
        with open(_f) as _fp:
            _d = json.load(_fp)
        _m = _d.get("median_val_ppl")
        if _m is not None:
            _sprint_medians[f"s{_s}_ppl"] = _m

if summary_file.exists():
    with open(summary_file) as f:
        summary = json.load(f)

    median_ppl = summary.get("median_val_ppl")
    print(f"Sprint {SPRINT}: {summary.get('description')}")
    print(f"  median val_ppl : {median_ppl}")
    print()

    results = summary.get("results", {})
    print(f"  {'Seed':<8} {'val_ppl':>10} {'elapsed':>10} {'status'}")
    print(f"  {'-'*45}")
    for seed_key, r in results.items():
        ppl = r.get("final_val_ppl")
        ppl_str = f"{ppl:.4f}" if ppl is not None else "n/a"
        status = "OK" if r.get("returncode") == 0 else f"FAILED rc={r.get('returncode')}"
        elapsed = r.get("elapsed_s", 0)
        print(f"  {seed_key:<8} {ppl_str:>10} {elapsed:>9.0f}s  {status}")

    if median_ppl is not None and SPRINT in _SPRINT_GATES:
        gate_desc, gate_fn = _SPRINT_GATES[SPRINT]
        passed = gate_fn(median_ppl, _sprint_medians)
        icon = "\u2705 PASS" if passed else "\u274c FAIL"
        print(f"\n  Gate : {gate_desc}")
        print(f"  Result: {icon}  (median={median_ppl:.4f})")
else:
    print(f"Summary file not found: {summary_file}")
    print("Check run output above for errors.")

---
## (Optional) Post-Training Evals

Passkey retrieval (Sprints 3–5) and BABILong (Sprint 5) evals run automatically below.  
These cells are no-ops for Sprints 1 and 2.

In [ ]:
# ── Passkey retrieval eval — Sprints 3, 4, 5 ─────────────────────────────────
# Max context window: 1024 for Sprints 3/4, 4096 for Sprint 5.
import glob as _glob

_SPRINT_NAMES = {1: "exp_poc_a", 2: "exp_poc_b", 3: "exp_poc_c", 4: "exp_poc_d", 5: "exp_poc_e"}
_MAX_CTX = {3: 1024, 4: 1024, 5: 4096}

if SPRINT in (3, 4, 5):
    name    = _SPRINT_NAMES[SPRINT]
    max_ctx = _MAX_CTX[SPRINT]
    seeds_for_sprint = [42] if SPRINT == 5 else SEEDS
    print(f"Passkey retrieval eval — Sprint {SPRINT}  (max_ctx={max_ctx})")

    for seed in seeds_for_sprint:
        ckpt_dir = f"{RESULTS_ROOT}/checkpoints/{name}_s{seed}"
        ckpts    = sorted(_glob.glob(f"{ckpt_dir}/*.safetensors"))
        if not ckpts:
            print(f"  Seed {seed}: no checkpoint found in {ckpt_dir} — skipping")
            continue
        ckpt     = ckpts[-1]
        log_path = f"{RESULTS_ROOT}/passkey_s{SPRINT}_s{seed}.log"
        print(f"  Seed {seed}: evaluating {ckpt}")

        r = subprocess.run(
            [sys.executable, "-m", "drex.eval.passkey",
             "--checkpoint", ckpt,
             "--max-context", str(max_ctx)],
            cwd=REPO_DIR,
            capture_output=True, text=True,
        )
        output = r.stdout + r.stderr
        with open(log_path, "w") as fh:
            fh.write(output)
        print(output[-2000:] if len(output) > 2000 else output)
        if r.returncode != 0:
            print(f"  WARN: passkey eval exited {r.returncode} — continuing")
else:
    print(f"Sprint {SPRINT}: passkey eval runs for Sprints 3, 4, 5 only — skipping.")

In [ ]:
# ── BABILong eval — Sprint 5 only ────────────────────────────────────────────
if SPRINT == 5:
    seeds_for_sprint = [42]
    print("BABILong eval — Sprint 5")

    for seed in seeds_for_sprint:
        ckpt_dir = f"{RESULTS_ROOT}/checkpoints/{_SPRINT_NAMES[5]}_s{seed}"
        ckpts    = sorted(_glob.glob(f"{ckpt_dir}/*.safetensors"))
        if not ckpts:
            print(f"  Seed {seed}: no checkpoint found in {ckpt_dir} — skipping")
            continue
        ckpt     = ckpts[-1]
        log_path = f"{RESULTS_ROOT}/babilong_s5_s{seed}.log"
        print(f"  Seed {seed}: evaluating {ckpt}")

        r = subprocess.run(
            [sys.executable, "-m", "drex.eval.babilong",
             "--checkpoint", ckpt],
            cwd=REPO_DIR,
            capture_output=True, text=True,
        )
        output = r.stdout + r.stderr
        with open(log_path, "w") as fh:
            fh.write(output)
        print(output[-2000:] if len(output) > 2000 else output)
        if r.returncode != 0:
            print(f"  WARN: babilong eval exited {r.returncode} — continuing")
else:
    print(f"Sprint {SPRINT}: BABILong eval runs for Sprint 5 only — skipping.")

---
## (Optional) Run all 5 sprints sequentially

Uncomment and run the cell below to queue the full campaign without changing
`SPRINT` between sessions. Sprint 5 will run after Sprints 1–4 have finished.
Suitable for an overnight / unattended Lightning AI run.

In [ ]:
# ── Run all 5 sprints via --all flag ──────────────────────────────────────────
# Uncomment the block below to run the full campaign.

# cmd_all = [
#     sys.executable,
#     os.path.join(REPO_DIR, "scripts", "poc", "run_poc_cloud.py"),
#     "--all",
#     "--out-dir", RESULTS_ROOT,
#     "--batch-size", str(BATCH_SIZE),
#     "--device", DEVICE,
#     "--repo-root", REPO_DIR,
# ]
# print("Running full campaign:", " ".join(cmd_all))
# result_all = subprocess.run(cmd_all, cwd=REPO_DIR)
# if result_all.returncode != 0:
#     raise RuntimeError(f"Full campaign exited with code {result_all.returncode}")
# print("Full campaign complete.")
print("Full-campaign cell is commented out. Remove the # above to enable.")